### Setup

In [0]:
%sql 
create catalog if not exists dbx;
create schema if not exists dbx.scenarios;

### ROW_NUMBER

In [0]:
%sql

Select employee_id, department_key, salary, row_number() over(partition by department_key order by salary) as rn from dbx.hr.dim_employee;

No Tie, No Gaps

###RANK

In [0]:
%sql
select employee_id, department_key, salary, rank() over(partition by department_key order by salary) as rnk from dbx.hr.dim_employee;

Allowed Ties, Allowed Gaps

###DENSE_RANK

In [0]:
%sql
select employee_id, department_key, salary, dense_rank() over(partition by department_key order by salary) as dn from dbx.hr.dim_employee;

Allowed Ties, No Gaps

###NTILE

In [0]:
%sql
select employee_id, department_key, salary, NTILE(2) over(partition by department_key order by salary) as buckets from dbx.hr.dim_employee;

In [0]:
%sql
select employee_id, department_key, salary, NTILE(3) over(partition by department_key order by salary) as buckets from dbx.hr.dim_employee;

Splits rows into n buckets as evenly as possible.

###Employees earning more than their managers

In [0]:
%sql
CREATE TABLE dbx.scenarios.employee (
    id INT PRIMARY KEY,
    name VARCHAR(100),
    salary INT,
    managerId INT
);

In [0]:
%sql
INSERT INTO dbx.scenarios.employee (id, name, salary, managerId) VALUES
(1, 'Joe',   70000, 3),
(2, 'Henry', 80000, 4),
(3, 'Sam',   60000, NULL),
(4, 'Max',   90000, NULL);

In [0]:
%sql
select e.name, e.salary from dbx.scenarios.employee e 
join dbx.scenarios.employee m
on e.managerId = m.id
where e.salary>m.salary;

###Managers with atleast 5 direct report

In [0]:
%sql
-- Create table
CREATE TABLE dbx.scenarios.employee_manager (
    id INT PRIMARY KEY,
    name VARCHAR(100),
    department VARCHAR(50),
    managerId INT
);


In [0]:
%sql
-- Insert sample data
INSERT INTO dbx.scenarios.employee_manager (id, name, department, managerId) VALUES
(101, 'John',  'A', NULL),
(102, 'Dan',   'A', 101),
(103, 'James', 'A', 101),
(104, 'Amy',   'A', 101),
(105, 'Anne',  'A', 101),
(106, 'Ron',   'B', 101);

In [0]:
%sql
select e.managerId, m.name, count(*) as manager from dbx.scenarios.employee_manager e 
join dbx.scenarios.employee_manager m
on e.managerId = m.id
group by 1,2
having count(*)>4


###Monthly sales aggregation with rows-to-columns transformation.

In [0]:
%sql
CREATE TABLE dbx.scenarios.salesdata (
    OrderID INT PRIMARY KEY,
    OrderDate DATE,
    SalesAmount DECIMAL(10,2)
);

In [0]:
%sql
INSERT INTO dbx.scenarios.salesdata (OrderID, OrderDate, SalesAmount) VALUES
(1,  '2023-01-15', 1000.00),
(2,  '2023-02-10', 1500.00),
(3,  '2023-03-05', 2000.00),
(4,  '2023-04-18', 1800.00),
(5,  '2023-05-22', 2200.00),
(6,  '2023-06-30', 2500.00),
(7,  '2023-07-12', 2700.00),
(8,  '2023-08-09', 3000.00),
(9,  '2023-09-14', 3200.00),
(10, '2023-10-03', 3500.00),
(11, '2023-11-19', 4000.00),
(12, '2023-12-25', 4500.00),
(13, '2024-01-08', 1100.00),
(14, '2024-02-16', 1600.00),
(15, '2024-03-21', 2100.00),
(16, '2024-04-11', 1900.00),
(17, '2024-05-27', 2300.00),
(18, '2024-06-05', 2600.00),
(19, '2024-07-17', 2800.00),
(20, '2024-08-29', 3100.00),
(21, '2024-09-02', 3300.00),
(22, '2024-10-15', 3600.00),
(23, '2024-11-23', 4100.00),
(24, '2024-12-31', 4700.00);

In [0]:
%sql
SELECT
    YEAR(OrderDate) AS SalesYear,

    SUM(CASE WHEN MONTH(OrderDate) = 1  THEN SalesAmount ELSE 0 END) AS Jan,
    SUM(CASE WHEN MONTH(OrderDate) = 2  THEN SalesAmount ELSE 0 END) AS Feb,
    SUM(CASE WHEN MONTH(OrderDate) = 3  THEN SalesAmount ELSE 0 END) AS Mar,
    SUM(CASE WHEN MONTH(OrderDate) = 4  THEN SalesAmount ELSE 0 END) AS Apr,
    SUM(CASE WHEN MONTH(OrderDate) = 5  THEN SalesAmount ELSE 0 END) AS May,
    SUM(CASE WHEN MONTH(OrderDate) = 6  THEN SalesAmount ELSE 0 END) AS Jun,
    SUM(CASE WHEN MONTH(OrderDate) = 7  THEN SalesAmount ELSE 0 END) AS Jul,
    SUM(CASE WHEN MONTH(OrderDate) = 8  THEN SalesAmount ELSE 0 END) AS Aug,
    SUM(CASE WHEN MONTH(OrderDate) = 9  THEN SalesAmount ELSE 0 END) AS Sep,
    SUM(CASE WHEN MONTH(OrderDate) = 10 THEN SalesAmount ELSE 0 END) AS Oct,
    SUM(CASE WHEN MONTH(OrderDate) = 11 THEN SalesAmount ELSE 0 END) AS Nov,
    SUM(CASE WHEN MONTH(OrderDate) = 12 THEN SalesAmount ELSE 0 END) AS Dec

FROM dbx.scenarios.salesdata
GROUP BY YEAR(OrderDate)
ORDER BY SalesYear;

In [0]:
%sql
SELECT *
FROM (
    SELECT
        YEAR(OrderDate) AS SalesYear,
        DATE_FORMAT(OrderDate, 'MMMM') AS SalesMonth,
        SalesAmount
    FROM dbx.scenarios.salesdata
) src
PIVOT (
    SUM(SalesAmount)
    FOR SalesMonth IN (
        'January','February','March','April','May','June',
        'July','August','September','October','November','December'
    )
)
ORDER BY SalesYear;

###Running total for different genders

In [0]:
%sql
CREATE TABLE dbx.scenarios.scores (
    player_name VARCHAR(100),
    gender VARCHAR(1),
    day DATE,
    score_points INT,
    PRIMARY KEY (gender, day, player_name)
);

In [0]:
%sql
INSERT INTO dbx.scenarios.scores (player_name, gender, day, score_points) VALUES
('Aron',     'F', '2020-01-01', 17),
('Alice',    'F', '2020-01-07', 23),
('Bajrang',  'M', '2020-01-07', 7),
('Khali',    'M', '2019-12-25', 11),
('Slaman',   'M', '2019-12-30', 13),
('Joe',      'M', '2019-12-31', 3),
('Jose',     'M', '2019-12-18', 2),
('Priya',    'F', '2019-12-31', 23),
('Priyanka', 'F', '2019-12-30', 17);

In [0]:
%sql
SELECT
    gender,
    day,
    SUM(score_points) OVER (
        PARTITION BY gender
        ORDER BY day
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS total
FROM dbx.scenarios.scores
ORDER BY gender, day;

In [0]:
%sql
SELECT
    gender,
    day,
    SUM(score_points) OVER (
        PARTITION BY gender
        ORDER BY day
        --ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS total
FROM dbx.scenarios.scores
ORDER BY gender, day;

###Longest Winning streak

In [0]:
%sql
CREATE TABLE dbx.scenarios.matches (
    player_id INT,
    match_day DATE,
    result STRING,
    PRIMARY KEY (player_id, match_day)
);

In [0]:
%sql
INSERT INTO dbx.scenarios.matches (player_id, match_day, result) VALUES
(1, '2022-01-17', 'Win'),
(1, '2022-01-18', 'Win'),
(1, '2022-01-25', 'Win'),
(1, '2022-01-31', 'Draw'),
(1, '2022-02-08', 'Win'),
(2, '2022-02-06', 'Lose'),
(2, '2022-02-08', 'Lose'),
(3, '2022-03-30', 'Win');

In [0]:
%sql
WITH ordered AS (
    SELECT *,
           CASE WHEN result = 'Win' THEN 1 ELSE 0 END AS is_win,
           ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY match_day) AS rn
    FROM dbx.scenarios.matches
),
grouped AS (
    SELECT *,
           rn - ROW_NUMBER() OVER (
                PARTITION BY player_id, is_win
                ORDER BY match_day
           ) AS grp
    FROM ordered
),
streaks AS (
    SELECT
        player_id,
        COUNT(*) AS streak
    FROM grouped
    WHERE is_win = 1
    GROUP BY player_id, grp
)
SELECT
    m.player_id,
    COALESCE(MAX(streak), 0) AS longest_streak
FROM dbx.scenarios.matches m
LEFT JOIN streaks s
    ON m.player_id = s.player_id
GROUP BY m.player_id;

###Active user retention by Month

In [0]:
%sql
CREATE TABLE dbx.scenarios.user_actions (
    user_id INT,
    event_id INT,
    event_type STRING,
    event_date TIMESTAMP
);

In [0]:
%sql
INSERT INTO dbx.scenarios.user_actions (user_id, event_id, event_type, event_date) VALUES
(445, 7765, 'sign-in', '2022-05-31 12:00:00'),
(742, 6458, 'sign-in', '2022-06-03 12:00:00'),
(445, 3634, 'like',    '2022-06-05 12:00:00'),
(742, 1374, 'comment', '2022-06-05 12:00:00'),
(648, 3124, 'like',    '2022-06-18 12:00:00');

In [0]:
%sql
WITH monthly_active AS (
    SELECT 
        user_id,
        YEAR(event_date)  AS year,
        MONTH(event_date) AS month
    FROM dbx.scenarios.user_actions
    GROUP BY 
        user_id,
        YEAR(event_date),
        MONTH(event_date)
)

SELECT 
    a.year,
    a.month                        AS current_month,
    COUNT(DISTINCT a.user_id)      AS retained_users
FROM monthly_active a
INNER JOIN monthly_active b
    ON  a.user_id = b.user_id
    AND (
        (a.month = b.month + 1 AND a.year = b.year)          -- same year, next month
        OR
        (a.month = 1 AND b.month = 12 AND a.year = b.year + 1) -- Jan of next year after Dec
    )
GROUP BY a.year, a.month
ORDER BY a.year, a.month;

###Year on Year growth rate

In [0]:
%sql
CREATE TABLE dbx.scenarios.user_transactions (
    transaction_id     INT,
    product_id         INT,
    spend              DECIMAL(10,2),
    transaction_date   TIMESTAMP
);

In [0]:
%sql
-- INSERT DATA (Example Input)

INSERT INTO dbx.scenarios.user_transactions
(transaction_id, product_id, spend, transaction_date)
VALUES
(1341, 123424, 1500.60, '2019-12-31 12:00:00'),
(1423, 123424, 1000.20, '2020-12-31 12:00:00'),
(1623, 123424, 1246.44, '2021-12-31 12:00:00'),
(1322, 123424, 2145.32, '2022-12-31 12:00:00');

In [0]:
%sql 
WITH yearly_spend AS (
    SELECT 
        YEAR(transaction_date)    AS year,
        product_id,
        SUM(spend)                AS curr_year_spend
    FROM dbx.scenarios.user_transactions
    GROUP BY 
        YEAR(transaction_date),
        product_id
)

SELECT 
    year,
    product_id,
    curr_year_spend,
    LAG(curr_year_spend) OVER (
        PARTITION BY product_id 
        ORDER BY year
    )                             AS prev_year_spend,
    ROUND(
        (curr_year_spend - LAG(curr_year_spend) OVER (
            PARTITION BY product_id ORDER BY year)
        ) 
        / LAG(curr_year_spend) OVER (
            PARTITION BY product_id ORDER BY year)
        * 100
    , 2)                          AS yoy_rate
FROM yearly_spend
ORDER BY product_id, year;

###Consecutive for 3 or more filling years

In [0]:
%sql
CREATE TABLE dbx.scenarios.filed_taxes (
    filing_id   INT,
    user_id     VARCHAR(50),
    filing_date TIMESTAMP,
    product     VARCHAR(100)
);

In [0]:
%sql
INSERT INTO dbx.scenarios.filed_taxes VALUES
(1, '1', '2019-04-14', 'TurboTax Desktop 2019'),
(2, '1', '2020-04-15', 'TurboTax Deluxe'),
(3, '1', '2021-04-15', 'TurboTax Online'),
(4, '2', '2020-04-07', 'TurboTax Online'),
(5, '2', '2021-04-10', 'TurboTax Online'),
(6, '3', '2020-04-07', 'TurboTax Online'),
(7, '3', '2021-04-15', 'TurboTax Online'),
(8, '3', '2022-03-11', 'QuickBooks Desktop Pro'),
(9, '4', '2022-04-15', 'QuickBooks Online');

In [0]:
%sql
WITH turbotax_years AS (
    SELECT 
        user_id,
        YEAR(filing_date) AS filing_year
    FROM dbx.scenarios.filed_taxes
    WHERE product LIKE '%TurboTax%'
),

consecutive_check AS (
    SELECT 
        user_id,
        filing_year,
        LAG(filing_year, 1) OVER (PARTITION BY user_id ORDER BY filing_year) AS prev_year_1,
        LAG(filing_year, 2) OVER (PARTITION BY user_id ORDER BY filing_year) AS prev_year_2
    FROM turbotax_years
)

SELECT DISTINCT user_id
FROM consecutive_check
WHERE filing_year - prev_year_1 = 1
  AND prev_year_1 - prev_year_2 = 1
ORDER BY user_id ASC;

### Gaps & Islands - Find all ISLANDS (Consecutive Streaks)

In [0]:
%sql
CREATE TABLE dbx.scenarios.logins (
    login_id   INT,
    user_id    INT,
    login_date DATE
);

In [0]:
%sql
INSERT INTO dbx.scenarios.logins VALUES
(1, 101, '2023-01-01'),
(2, 101, '2023-01-02'),
(3, 101, '2023-01-03'),
(4, 101, '2023-01-06'),
(5, 101, '2023-01-07'),
(6, 101, '2023-01-10');

In [0]:
%sql 
WITH ranked AS (
    SELECT 
        user_id,
        login_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id 
            ORDER BY login_date
        )                                         AS row_num
    FROM dbx.scenarios.logins
),

island_groups AS (
    SELECT 
        user_id,
        login_date,
        -- Databricks SQL syntax: use DATEADD or direct subtraction
        DATEADD(DAY, -row_num, login_date)        AS grp
    FROM ranked
)

SELECT 
    user_id,
    MIN(login_date)  AS island_start,
    MAX(login_date)  AS island_end,
    COUNT(*)         AS consecutive_days
FROM island_groups
GROUP BY user_id, grp
ORDER BY island_start;

### Gaps & Islands - Find all GAPS (Missing Date Ranges)

In [0]:
%sql
WITH ranked AS (
    SELECT 
        user_id,
        login_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id 
            ORDER BY login_date
        )                                          AS row_num
    FROM dbx.scenarios.logins
),

island_groups AS (
    SELECT 
        user_id,
        login_date,
        DATEADD(DAY, -row_num, login_date)         AS grp
    FROM ranked
),

islands AS (
    SELECT 
        user_id,
        MIN(login_date)                            AS island_start,
        MAX(login_date)                            AS island_end
    FROM island_groups
    GROUP BY user_id, grp
),

gaps AS (
    SELECT 
        user_id,
        DATEADD(DAY, 1, island_end)                AS gap_start,
        DATEADD(DAY, -1, LEAD(island_start) OVER (
            PARTITION BY user_id 
            ORDER BY island_start)
        )                                          AS gap_end
    FROM islands
)

SELECT 
    user_id,
    gap_start,
    gap_end
FROM gaps
WHERE gap_start <= gap_end   -- removes NULLs and invalid gaps
ORDER BY user_id, gap_start;

### Gaps & Islands - Longest Consecutive Streak per User

In [0]:
%sql
WITH ranked AS (
    SELECT 
        user_id,
        login_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id 
            ORDER BY login_date
        )                                  AS row_num
    FROM dbx.scenarios.logins
),

island_groups AS (
    SELECT 
        user_id,
        login_date,
        DATEADD(DAY, -row_num, login_date) AS grp   -- Databricks syntax
    FROM ranked
),

streaks AS (
    SELECT 
        user_id,
        COUNT(*)                           AS streak_length
    FROM island_groups
    GROUP BY user_id, grp
)

SELECT 
    user_id,
    MAX(streak_length)                     AS longest_streak
FROM streaks
GROUP BY user_id
ORDER BY longest_streak DESC;

###Find a sale person id who have made sales for 3 or more consecutive days

In [0]:
%sql
CREATE TABLE dbx.scenarios.salesbyperson (
    sale_id         INT,
    salesperson_id  INT,
    sale_date       DATE
);

In [0]:
%sql
-- INSERT SAMPLE DATA

INSERT INTO dbx.scenarios.salesbyperson (sale_id, salesperson_id, sale_date) VALUES
-- Salesperson 101 → has 3 consecutive days (qualifies)
(1, 101, '2023-01-01'),
(2, 101, '2023-01-02'),
(3, 101, '2023-01-03'),
(4, 101, '2023-01-06'),

-- Salesperson 102 → only 2 consecutive days (does NOT qualify)
(5, 102, '2023-02-01'),
(6, 102, '2023-02-02'),
(7, 102, '2023-02-05'),

-- Salesperson 103 → 4 consecutive days (qualifies)
(8, 103, '2023-03-10'),
(9, 103, '2023-03-11'),
(10,103, '2023-03-12'),
(11,103, '2023-03-13'),

-- Salesperson 104 → non-consecutive sales (does NOT qualify)
(12,104, '2023-04-01'),
(13,104, '2023-04-03'),
(14,104, '2023-04-05');

In [0]:
%sql 
WITH ranked_sales AS (
    SELECT 
        salesperson_id,
        sale_date,
        ROW_NUMBER() OVER (
            PARTITION BY salesperson_id 
            ORDER BY sale_date
        )                                      AS row_num
    FROM dbx.scenarios.salesbyperson
    GROUP BY salesperson_id, sale_date         -- one record per day per salesperson
),

grouping_logic AS (
    SELECT 
        salesperson_id,
        sale_date,
        DATEADD(DAY, -row_num, sale_date)      AS grp   -- ✅ Databricks syntax
    FROM ranked_sales
),

consecutive_counts AS (
    SELECT 
        salesperson_id,
        grp,
        MIN(sale_date)                         AS streak_start,
        MAX(sale_date)                         AS streak_end,
        COUNT(*)                               AS consecutive_days
    FROM grouping_logic
    GROUP BY salesperson_id, grp
)

SELECT DISTINCT salesperson_id
FROM consecutive_counts
WHERE consecutive_days >= 3
ORDER BY salesperson_id;